In [ ]:
"""
AEC + LogReg baseline for PD vs HC binary classification on baseline MEG.
 
Pipeline:
  1. Load source-reconstructed time series
  2. Compute pairwise Amplitude Envelope Correlation (AEC) per subject
  3. Stratified k-fold CV with subject-level splits, all preprocessing inside fold
  4. Train LogReg L2 and Linear SVM
  5. Report metrics + permutation test for significance
 
Adjust BASE_PATH to point to your local data directory.
"""

### 0. Setup

In [29]:
import os
from pathlib import Path
 
import numpy as np
import pandas as pd
from scipy.signal import hilbert, butter, sosfiltfilt, decimate
 
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    roc_auc_score, accuracy_score,
    precision_score, recall_score, f1_score,
)
 
# used for logging
import sys, time
from datetime import datetime
from pathlib import Path

In [3]:
# ----------------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------------
 
BASE_PATH = Path("/Users/georg/kth/MEG_project/data/FullData") 
META_FILE = BASE_PATH / "pd_longitudinal_metadata_short.xlsx"
DATA_ROOT = BASE_PATH / "CIR_PD_wholebrain_ts"
BL_FOLDER = DATA_ROOT / "wholebrain_ts_aparc_dSPM_bl"
 
SAMPLING_RATE = 1000  # Hz
USE_FIRST_N_SECONDS = None  # None = use full recording; set to 180 for 3-min pilot

In [10]:
# ----------------------------------------------------------------------------
# LOGGING SETUP
# ----------------------------------------------------------------------------

LOG_DIR = Path("logs")   # created relative to the notebook's working dir;                        


class _Tee:
    """Write to several streams at once (console + file)."""
    def __init__(self, *streams):
        self.streams = streams
    def write(self, data):
        for s in self.streams:
            s.write(data)
    def flush(self):
        for s in self.streams:
            s.flush()

### 1. Data loading

In [4]:
# ----------------------------------------------------------------------------
# DATA LOADING
# ----------------------------------------------------------------------------
 
def load_metadata():
    """Load the metadata Excel and build subject → label mapping."""
    df = pd.read_excel(META_FILE, sheet_name="Sheet1")
    # group is "pd" or "hc"
    df["label"] = (df["group"].str.lower() == "pd").astype(int)
    return df
 
 
def build_subject_mapping(df_meta):
    """Build NatMEG_XXXX (baseline) → row in metadata."""
    mapping = {}
    for _, row in df_meta.iterrows():
        bl_key = f"NatMEG_{int(row['id_bl']):04d}"
        mapping[bl_key] = row
    return mapping
 
 
def list_baseline_subjects():
    """List subject folders in baseline directory."""
    subjects = [d for d in os.listdir(BL_FOLDER) if d.startswith("NatMEG_")]
    subjects.sort()
    return subjects
 
 
def load_subject_data(subject_dir):
    """
    Load all 68 ROI .npy files for one subject.
 
    Returns:
        data: (n_roi, n_timepoints) array
        roi_names: list of length n_roi with filename stems
    """
    files = sorted(os.listdir(subject_dir))
    files = [f for f in files if f.endswith(".npy")]
    arrays = []
    roi_names = []
    for fname in files:
        arr = np.load(os.path.join(subject_dir, fname))
        arrays.append(arr)
        # filename like "0739-transversetemporal-rh.npy" → roi name
        stem = fname.replace(".npy", "")
        parts = stem.split("-", 1)  # drop subject ID prefix
        roi_names.append(parts[1] if len(parts) > 1 else stem)
    return np.array(arrays), roi_names
 
 
def load_all_subjects(verbose=True):
    """
    Load every baseline subject that exists on disk AND in the metadata.
 
    Returns:
        X_raw:       list of (n_roi, n_timepoints) arrays, one per subject
        y:           (n_subjects,) array of labels (1=PD, 0=HC)
        subject_ids: list of NatMEG_XXXX strings
        roi_names:   list of ROI names (same order for all subjects)
    """
    df_meta = load_metadata()
    mapping = build_subject_mapping(df_meta)
    subjects_on_disk = list_baseline_subjects()
 
    X_raw, y, subject_ids = [], [], []
    roi_names_ref = None
 
    skipped = []
    for subj in subjects_on_disk:
        if subj not in mapping:
            skipped.append((subj, "not in metadata"))
            continue
        row = mapping[subj]
        subj_dir = BL_FOLDER / subj
        data, roi_names = load_subject_data(subj_dir)
 
        # Sanity check: ROI ordering consistent across subjects
        if roi_names_ref is None:
            roi_names_ref = roi_names
        else:
            if roi_names != roi_names_ref:
                skipped.append((subj, "ROI order mismatch"))
                continue
 
        # Sanity check: 68 ROIs expected
        if data.shape[0] != 68:
            skipped.append((subj, f"wrong n_roi={data.shape[0]}"))
            continue
 
        X_raw.append(data)
        y.append(int(row["label"]))
        subject_ids.append(subj)
 
    if verbose:
        print(f"Loaded {len(X_raw)} subjects "
              f"({sum(y)} PD, {len(y) - sum(y)} HC)")
        if skipped:
            print(f"Skipped {len(skipped)}:")
            for s, reason in skipped:
                print(f"  {s}: {reason}")
 
    return X_raw, np.array(y), subject_ids, roi_names_ref

### 2. Feature Extraction

In [5]:
# ----------------------------------------------------------------------------
# FEATURE EXTRACTION: AEC
# ----------------------------------------------------------------------------


def compute_aec(data):
    """
    Amplitude Envelope Correlation.
 
    Args:
        data: (n_roi, n_timepoints) array of source time series
    Returns:
        aec: (n_roi, n_roi) symmetric correlation matrix
    """
    # Hilbert transform per ROI → amplitude envelope
    analytic = hilbert(data, axis=1)
    envelope = np.abs(analytic)
    # Pearson correlation of envelopes
    aec = np.corrcoef(envelope)
    return aec


def matrix_to_features(M):
    """Upper-triangular (excluding diagonal) of a symmetric matrix → 1D vector."""
    iu = np.triu_indices(M.shape[0], k=1)
    return M[iu]


def build_feature_matrix(X_raw, max_samples=None):
    """
    Compute AEC features for every subject.
 
    Args:
        X_raw: list of (n_roi, n_timepoints) arrays
        max_samples: if set, truncate each recording to this many samples
                     (e.g. SAMPLING_RATE * 180 for first 3 minutes)
    Returns:
        X: (n_subjects, n_features) array, n_features = 68*67/2 = 2278
    """
    feats = []
    for data in X_raw:
        if max_samples is not None:
            data = data[:, :max_samples]
        aec = compute_aec(data)
        feats.append(matrix_to_features(aec))
    return np.array(feats)

In [30]:
# ============================================================================
# Band-resolved, leakage-corrected AEC  (drop-in replacement for compute_aec)
# ----------------------------------------------------------------------------
# Two fixes vs the broadband version:
#   1. Band-pass -> Hilbert -> envelope, PER BAND. The analytic signal is only
#      well-defined for a narrowband input, so broadband AEC was smearing every
#      rhythm together. Now each band is resolved separately (beta is the one
#      to watch for PD).
#   2. Pairwise orthogonalization (Hipp et al. 2012), symmetrized. Removes the
#      zero-lag, same-phase component before correlating envelopes, which is
#      what the inverse operator injects as spurious connectivity. Same for PD
#      and HC, so leaving it in only dilutes class signal.
#
# Reuses matrix_to_features / evaluate / report / permutation_test / the
# pipelines UNCHANGED. Only the feature construction differs.
# ============================================================================
 
from scipy.signal import butter, sosfiltfilt, hilbert
 
# Canonical bands (Hz)
BANDS = {
    "delta": (1.0, 4.0),
    "theta": (4.0, 8.0),
    "alpha": (8.0, 13.0),
    "beta":  (13.0, 30.0),
    "gamma": (30.0, 45.0),
}
 
 
def _bandpass(data, fs, lo, hi, order=4):
    """Zero-phase Butterworth band-pass along time (axis=1)."""
    sos = butter(order, [lo, hi], btype="band", fs=fs, output="sos")
    return sosfiltfilt(sos, data, axis=1)
 
 
def _corr_rows_with(env_mat, ref_env):
    """
    Pearson corr of each row of env_mat (n_roi, n_time) with ref_env (n_time,).
    Vectorized; returns (n_roi,). Diagonal (self vs self) yields nan here and is
    overwritten to 0 by the caller.
    """
    a = env_mat - env_mat.mean(axis=1, keepdims=True)
    b = ref_env - ref_env.mean()
    num = a @ b
    den = np.sqrt((a ** 2).sum(axis=1) * (b ** 2).sum())
    with np.errstate(invalid="ignore", divide="ignore"):
        return num / den
 
 
def compute_aec_orth(data, fs, band, target_fs=200):
    """Orthogonalized AEC for one band, with post-bandpass downsampling."""
    lo, hi = band
    x = _bandpass(data, fs, lo, hi)          # narrowband at original fs

    # Downsample: band is well below the new Nyquist, so this is lossless
    # for our purpose and cuts the Hilbert cost by ~fs/target_fs.
    if target_fs is not None and target_fs < fs:
        factor = int(round(fs / target_fs))
        if factor > 1:
            x = decimate(x, factor, axis=1, ftype="fir")

    A = hilbert(x, axis=1)
    env = np.abs(A)
    n_roi = A.shape[0]

    M = np.zeros((n_roi, n_roi))
    for j in range(n_roi):
        ref = A[j] / np.abs(A[j])
        R = np.imag(A * np.conj(ref))
        env_orth = np.abs(hilbert(R, axis=1))
        M[:, j] = _corr_rows_with(env_orth, env[j])

    np.fill_diagonal(M, 0.0)
    return 0.5 * (M + M.T)
 
 
def build_band_feature_matrices(X_raw, fs, bands=None, max_samples=None,
                                 matrix_to_features=None, verbose=True):
    """
    Compute orthogonalized AEC features for every subject, per band.
 
    Args:
        X_raw:   list of (n_roi, n_time) arrays, one per subject
        fs:      sampling rate (Hz), e.g. SAMPLING_RATE
        bands:   dict name -> (lo, hi). Defaults to BANDS. Pass a 1-entry dict
                 (e.g. {"beta": BANDS["beta"]}) to run a single band fast.
        max_samples: truncate each recording to this many samples (mirrors
                     build_feature_matrix; e.g. fs * 180 for the 3-min pilot)
        matrix_to_features: pass your existing notebook function. If None, uses
                     the upper triangle (k=1), same as your current code.
    Returns:
        dict: band name -> (n_subjects, n_features) feature matrix
    """
    if bands is None:
        bands = BANDS
    if matrix_to_features is None:
        def matrix_to_features(M):
            iu = np.triu_indices(M.shape[0], k=1)
            return M[iu]
 
    out = {b: [] for b in bands}
    for s_idx, data in enumerate(X_raw):
        if max_samples is not None:
            data = data[:, :max_samples]
        for bname, band in bands.items():
            aec = compute_aec_orth(data, fs, band)
            out[bname].append(matrix_to_features(aec))
        if verbose and (s_idx + 1) % 10 == 0:
            print(f"  {s_idx + 1}/{len(X_raw)} subjects done")
 
    return {b: np.array(v) for b, v in out.items()}

### 3. Training setup

In [31]:
# ----------------------------------------------------------------------------
# CROSS-VALIDATION
# ----------------------------------------------------------------------------
 
def make_logreg_pipeline(C=1.0):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            penalty="l2",
            C=C,
            solver="liblinear",
            max_iter=5000,
            class_weight="balanced",
        )),
    ])
 
    
def make_svm_pipeline(C=1.0):
    # LinearSVC has no predict_proba; wrap with CalibratedClassifierCV for AUC.
    base_svm = LinearSVC(
        C=C, max_iter=10000, class_weight="balanced", dual=True,
    )
    return Pipeline([
        ("scaler", StandardScaler()),
        ("clf", CalibratedClassifierCV(base_svm, cv=3, method="sigmoid")),
    ])

def evaluate(X, y, pipe_factory, n_splits=5, n_seeds=20):
    """
    Stratified k-fold CV, repeated across seeds.
 
    All preprocessing happens inside the pipeline, so it's refit per fold —
    no leakage from scaler statistics or any other parameter.
 
    Returns dict of metric → array of per-fold values.
    """
    metrics = {k: [] for k in
               ["auc", "acc", "sens", "spec", "f1"]}
 
    for seed in range(n_seeds):
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        for tr, te in skf.split(X, y):
            pipe = pipe_factory()
            pipe.fit(X[tr], y[tr])
            y_pred = pipe.predict(X[te])
            y_prob = pipe.predict_proba(X[te])[:, 1]
 
            metrics["auc"].append(roc_auc_score(y[te], y_prob))
            metrics["acc"].append(accuracy_score(y[te], y_pred))
            metrics["sens"].append(recall_score(y[te], y_pred))
            # specificity = recall of negative class
            metrics["spec"].append(
                recall_score(y[te], y_pred, pos_label=0)
            )
            metrics["f1"].append(f1_score(y[te], y_pred))
 
    return {k: np.array(v) for k, v in metrics.items()}
 
 
def report(metrics, name):
    print(f"\n=== {name} ===")
    for k, v in metrics.items():
        print(f"  {k:5s}: {v.mean():.3f} ± {v.std():.3f}")

In [41]:
# ----------------------------------------------------------------------------
# PERMUTATION TEST
# ----------------------------------------------------------------------------
 
def permutation_test(X, y, pipe_factory,
                     n_perms=1000, n_splits=5, n_seeds=3, seed=42):
    """
    Shuffle labels and re-run CV. The fraction of null AUCs >= observed AUC
    is the p-value.
 
    Note: n_seeds is lower here (3 not 20) per permutation to keep total
    runtime manageable. 1000 perms × 3 seeds × 5 folds = 15,000 fits.
    """
    print(f"\nRunning permutation test ({n_perms} perms)...")
 
    # Observed: full evaluation
    obs = evaluate(X, y, pipe_factory, n_splits=n_splits, n_seeds=n_seeds)
    obs_auc = obs["auc"].mean()
 
    rng = np.random.default_rng(seed)
    null_aucs = np.empty(n_perms)
    for i in range(n_perms):
        y_shuf = rng.permutation(y)
        null = evaluate(X, y_shuf, pipe_factory,
                        n_splits=n_splits, n_seeds=1)
        null_aucs[i] = null["auc"].mean()
        if (i + 1) % 100 == 0:
            print(f"  {i+1}/{n_perms} (current null mean = "
                  f"{null_aucs[:i+1].mean():.3f})")
 
    p_value = (null_aucs >= obs_auc).mean()
    print(f"\nObserved AUC: {obs_auc:.3f}")
    print(f"Null AUC: {null_aucs.mean():.3f} ± {null_aucs.std():.3f}")
    print(f"Permutation p-value: {p_value:.4f}")
    return obs_auc, null_aucs, p_value
 

### 4. RUN

In [42]:
# ----------------------------------------------------------------------------
# MAIN  (fixed AEC: band-resolved, leakage-corrected)
# ----------------------------------------------------------------------------
FS = SAMPLING_RATE

def main():
    print("Loading data...")
    X_raw, y, subject_ids, roi_names = load_all_subjects()
    print(f"ROI count: {len(roi_names)}")
    print(f"First few ROIs: {roi_names[:3]}")

    # === All bands (3-min) ===
    print("\n--- orth-AEC all bands (3-min) ---")
    Xall = build_band_feature_matrices(
        X_raw, FS, bands=BANDS, max_samples=FS * 180,
        matrix_to_features=matrix_to_features,
    )
    for bname, Xband in Xall.items():
        m = evaluate(Xband, y, make_logreg_pipeline)
        report(m, f"3-MIN | orth-AEC {bname} | LogReg")

    return Xall, y, subject_ids, roi_names

In [43]:
def run_with_logging():
    LOG_DIR.mkdir(parents=True, exist_ok=True)
    start = datetime.now()
    stamp = start.strftime("%d-%m-%Y_%H%M%S")
    log_path = LOG_DIR / f"aec_run_{stamp}.log"

    t0 = time.perf_counter()
    old_stdout = sys.stdout
    result = None
    with open(log_path, "w", encoding="utf-8") as f:
        sys.stdout = _Tee(old_stdout, f)
        try:
            print(f"Run started:  {start:%Y-%m-%d %H:%M:%S}")
            print(f"Log file:     {log_path}")
            print("=" * 60)
            result = main()
        finally:
            elapsed = time.perf_counter() - t0
            print("=" * 60)
            print(f"Run finished: {datetime.now():%Y-%m-%d %H:%M:%S}")
            print(f"Duration:     {elapsed:.1f} s  ({elapsed/60:.1f} min)")
            sys.stdout = old_stdout

    print(f"Saved log to {log_path}")
    return result

In [44]:
Xall, y, subject_ids, roi_names = run_with_logging()

Run started:  2026-06-10 14:06:22
Log file:     logs\aec_run_10-06-2026_140622.log
Loading data...
Loaded 58 subjects (27 PD, 31 HC)
Skipped 1:
  NatMEG_0626: not in metadata
ROI count: 68
First few ROIs: ['bankssts-lh', 'bankssts-rh', 'caudalanteriorcingulate-lh']

--- orth-AEC all bands (3-min) ---
  10/58 subjects done
  20/58 subjects done
  30/58 subjects done
  40/58 subjects done
  50/58 subjects done

=== 3-MIN | orth-AEC delta | LogReg ===
  auc  : 0.643 ± 0.146
  acc  : 0.588 ± 0.126
  sens : 0.554 ± 0.229
  spec : 0.613 ± 0.216
  f1   : 0.540 ± 0.166

=== 3-MIN | orth-AEC theta | LogReg ===
  auc  : 0.549 ± 0.147
  acc  : 0.543 ± 0.132
  sens : 0.502 ± 0.233
  spec : 0.577 ± 0.209
  f1   : 0.488 ± 0.180

=== 3-MIN | orth-AEC alpha | LogReg ===
  auc  : 0.743 ± 0.134
  acc  : 0.675 ± 0.115
  sens : 0.616 ± 0.218
  spec : 0.725 ± 0.166
  f1   : 0.623 ± 0.159

=== 3-MIN | orth-AEC beta | LogReg ===
  auc  : 0.631 ± 0.138
  acc  : 0.585 ± 0.115
  sens : 0.514 ± 0.244
  spec : 0.

### 5. Permutation Test

In [46]:
# ----------------------------------------------------------------------------
# Max-over-bands permutation test (family-wise corrected p for the best band)
# ----------------------------------------------------------------------------
# Scanned 5 bands and picked alpha. Testing alpha against its own shuffled
# null ignores that we have looked at 5. The correct null is the MAX AUC across all
# bands per label shuffle — that accounts for the band selection.
# Runs on precomputed Xall, so it's fast (no AEC recompute).

def evaluate_auc(X, y, n_seeds=3):
    return evaluate(X, y, make_logreg_pipeline, n_seeds=n_seeds)["auc"].mean()

def permutation_test_maxband(Xall, y, bands=BANDS, n_perms=1000, n_seeds=3, seed=42):
    # observed AUC per band on the real labels
    obs = {b: evaluate_auc(Xall[b], y, n_seeds) for b in bands}
    best = max(obs, key=obs.get)
    print("Observed per-band AUC:")
    for b, a in sorted(obs.items(), key=lambda kv: -kv[1]):
        print(f"  {b:6s}: {a:.3f}")
    print(f"Best band: {best} ({obs[best]:.3f})\n")

    # null: for each shuffle, take the MAX AUC across all bands
    rng = np.random.default_rng(seed)
    null_max = np.empty(n_perms)
    for k in range(n_perms):
        ys = rng.permutation(y)
        null_max[k] = max(evaluate_auc(Xall[b], ys, n_seeds=1) for b in bands)
        if (k + 1) % 100 == 0:
            print(f"  {k+1}/{n_perms} (null max mean = {null_max[:k+1].mean():.3f})")

    p_corr = (null_max >= obs[best]).mean()
    print(f"\nFamily-wise corrected p ({best} vs {len(bands)}-band null): {p_corr:.4f}")
    print(f"Null max AUC: {null_max.mean():.3f} ± {null_max.std():.3f}")
    return obs, best, null_max, p_corr

obs, best, null_max, p_corr = permutation_test_maxband(Xall, y, n_perms=1000)

Observed per-band AUC:
  alpha : 0.761
  delta : 0.650
  beta  : 0.636
  theta : 0.549
  gamma : 0.508
Best band: alpha (0.761)

  100/1000 (null max mean = 0.614)
  200/1000 (null max mean = 0.603)
  300/1000 (null max mean = 0.605)
  400/1000 (null max mean = 0.604)
  500/1000 (null max mean = 0.604)
  600/1000 (null max mean = 0.602)
  700/1000 (null max mean = 0.603)
  800/1000 (null max mean = 0.603)
  900/1000 (null max mean = 0.605)
  1000/1000 (null max mean = 0.604)

Family-wise corrected p (alpha vs 5-band null): 0.0180
Null max AUC: 0.604 ± 0.079


### Save features

In [49]:
from pathlib import Path

FEAT_PATH = BASE_PATH / "logs" / "features" / "orthAEC_3min_allbands_6-10-2026.npz"
FEAT_PATH.parent.mkdir(parents=True, exist_ok=True)

np.savez_compressed(
    FEAT_PATH,
    y=y,
    roi_names=np.array(roi_names),
    subject_ids=np.array(subject_ids),
    max_samples=np.array(FS * 180),
    **Xall,
)
print(f"Saved {FEAT_PATH}  ({FEAT_PATH.stat().st_size/1e6:.1f} MB)")

Saved \Users\georg\kth\MEG_project\data\FullData\logs\features\orthAEC_3min_allbands_6-10-2026.npz  (5.0 MB)


### Reload features

In [ ]:
d = np.load(FEAT_PATH, allow_pickle=False)
Xall = {b: d[b] for b in BANDS}
y = d["y"]
roi_names = d["roi_names"].tolist()
subject_ids = d["subject_ids"].tolist()
print(f"Loaded {len(y)} subjects, bands: {list(Xall.keys())}")

In [50]:
# ----------------------------------------------------------------------------
# Alpha A–E localization sweep — WHERE does the alpha PD signal live?
# Runs on precomputed Xall["alpha"] (seconds, no AEC recompute).
#   within    = edges among the kept ROIs  (Erik's "keep these channels")
#   involving = edges from the kept ROIs to the whole brain (seed connectivity)
# Needs: DK_REGION_SETS, select_roi_indices, edge_mask, SENSORY_CONTROLS, evaluate_auc
# ----------------------------------------------------------------------------
Xa = Xall["alpha"]
full_auc = evaluate_auc(Xa, y, n_seeds=20)
print(f"Reference: all 68 ROIs, {Xa.shape[1]} edges | alpha AUC {full_auc:.3f}\n")

print(f"{'set':<4}{'n_roi':>6}{'within':>20}{'involving':>20}")
print("-" * 50)
for s in ["A", "B", "C", "D", "E"]:
    idx = select_roi_indices(roi_names, DK_REGION_SETS[s])

    mw = edge_mask(roi_names, idx, mode="within")
    if mw.sum() > 0:
        within_str = f"{mw.sum():4d} ed  AUC {evaluate_auc(Xa[:, mw], y, n_seeds=20):.3f}"
    else:
        within_str = "1 ROI pair (skip)"

    mi = edge_mask(roi_names, idx, mode="involving")
    involving_str = f"{mi.sum():4d} ed  AUC {evaluate_auc(Xa[:, mi], y, n_seeds=20):.3f}"

    print(f"{s:<4}{len(idx):>6}{within_str:>20}{involving_str:>20}")

# Negative control: PD-spared sensory regions (V1/A1/S1), within-set edges
sidx = select_roi_indices(roi_names, sum(SENSORY_CONTROLS.values(), []))
sm = edge_mask(roi_names, sidx, mode="within")
print("-" * 50)
print(f"{'ctrl':<4}{len(sidx):>6}{f'{sm.sum():4d} ed  AUC {evaluate_auc(Xa[:, sm], y, n_seeds=20):.3f}':>20}"
      f"{'(V1/A1/S1, expect ~0.5)':>20}")

Reference: all 68 ROIs, 2278 edges | alpha AUC 0.743

set  n_roi              within           involving
--------------------------------------------------
A        2     1 ed  AUC 0.554   133 ed  AUC 0.566
B       14    91 ed  AUC 0.671   847 ed  AUC 0.757
C       22   231 ed  AUC 0.716  1243 ed  AUC 0.767
D       28   378 ed  AUC 0.725  1498 ed  AUC 0.752
E       68  2278 ed  AUC 0.743  2278 ed  AUC 0.743
--------------------------------------------------
ctrl     6    15 ed  AUC 0.710(V1/A1/S1, expect ~0.5)


<br><br><br><br><br><br><br><br><br><br><br><br>

In [33]:
# # ----------------------------------------------------------------------------
# # MAIN  (fixed AEC: band-resolved, leakage-corrected)
# # ----------------------------------------------------------------------------

# FS = SAMPLING_RATE


# def main():
#     print("Loading data...")
#     X_raw, y, subject_ids, roi_names = load_all_subjects()
#     print(f"ROI count: {len(roi_names)}")
#     print(f"First few ROIs: {roi_names[:3]}")

#     # === All bands: AUC sweep only (cheap) ===
#     print("\n--- orth-AEC all bands (3-min) ---")
#     Xall = build_band_feature_matrices(
#         X_raw, FS, bands=BANDS, max_samples=FS * 180,
#         matrix_to_features=matrix_to_features,
#     )

#     aucs = {}
#     for bname, Xband in Xall.items():
#         m = evaluate(Xband, y, make_logreg_pipeline)
#         report(m, f"3-MIN | orth-AEC {bname} | LogReg")
#         aucs[bname] = m["auc"].mean()

#     # === Permutation test on the single best band ===
#     best = max(aucs, key=aucs.get)
#     print(f"\n--- Permutation test on best band: {best} "
#           f"(AUC {aucs[best]:.3f}) ---")
#     permutation_test(Xall[best], y, make_logreg_pipeline, n_perms=1000)

#     return Xall

In [ ]:
Xall, y, subject_ids, roi_names = run_with_logging()

In [35]:
Xall = run_with_logging()

Run started:  2026-06-10 11:24:47
Log file:     logs\aec_run_10-06-2026_112447.log
Loading data...
Loaded 58 subjects (27 PD, 31 HC)
Skipped 1:
  NatMEG_0626: not in metadata
ROI count: 68
First few ROIs: ['bankssts-lh', 'bankssts-rh', 'caudalanteriorcingulate-lh']

--- orth-AEC all bands (3-min) ---
  10/58 subjects done
  20/58 subjects done
  30/58 subjects done
  40/58 subjects done
  50/58 subjects done

=== 3-MIN | orth-AEC delta | LogReg ===
  auc  : 0.643 ± 0.146
  acc  : 0.588 ± 0.126
  sens : 0.554 ± 0.229
  spec : 0.613 ± 0.216
  f1   : 0.540 ± 0.166

=== 3-MIN | orth-AEC theta | LogReg ===
  auc  : 0.549 ± 0.147
  acc  : 0.543 ± 0.132
  sens : 0.502 ± 0.233
  spec : 0.577 ± 0.209
  f1   : 0.488 ± 0.180

=== 3-MIN | orth-AEC alpha | LogReg ===
  auc  : 0.743 ± 0.134
  acc  : 0.675 ± 0.115
  sens : 0.616 ± 0.218
  spec : 0.725 ± 0.166
  f1   : 0.623 ± 0.159

=== 3-MIN | orth-AEC beta | LogReg ===
  auc  : 0.631 ± 0.138
  acc  : 0.585 ± 0.115
  sens : 0.514 ± 0.244
  spec : 0.

In [39]:
# ----------------------------------------------------------------------------
# Channel-selection scheme (A–E) → Desikan-Killiany aparc labels
# ----------------------------------------------------------------------------
DK_REGION_SETS = {
    # A: primary motor cortex (M1) = precentral gyrus
    "A": ["precentral"],
    # B: A + SMA/pre-SMA + PFC (SFG, MFG, IFG) + PMC (=precentral, collapses into A)
    "B": ["precentral", "superiorfrontal",
          "rostralmiddlefrontal", "caudalmiddlefrontal",     # middle frontal gyrus
          "parsopercularis", "parstriangularis", "parsorbitalis"],  # inferior frontal gyrus
}
# C: B + inferior parietal (IPL) + superior parietal (SPL) + posterior parietal
DK_REGION_SETS["C"] = DK_REGION_SETS["B"] + [
    "inferiorparietal", "supramarginal", "superiorparietal", "precuneus"]
# D: C + superior / middle / inferior temporal gyrus
DK_REGION_SETS["D"] = DK_REGION_SETS["C"] + [
    "superiortemporal", "middletemporal", "inferiortemporal"]
# E: all 68
DK_REGION_SETS["E"] = "ALL"

# Tissue-spared sensory regions (Wiesman 2024) for the control-distance check
SENSORY_CONTROLS = {"V1": ["pericalcarine"],
                    "A1": ["transversetemporal"],   # Heschl's gyrus
                    "S1": ["postcentral"]}

def roi_base(name):
    return name.rsplit("-", 1)[0]

def select_roi_indices(roi_names, labels):
    if labels == "ALL":
        return list(range(len(roi_names)))
    labset = set(labels)
    return [i for i, n in enumerate(roi_names) if roi_base(n) in labset]

def edge_mask(roi_names, sel_idx, mode="within"):
    """Boolean mask over upper-triangle (k=1) edges, same order as matrix_to_features.
       mode='within'    : both endpoints in the set
       mode='involving' : at least one endpoint in the set (M1's connectivity to all)"""
    n = len(roi_names)
    i, j = np.triu_indices(n, k=1)
    sel = set(sel_idx)
    if mode == "within":
        return np.array([(a in sel) and (b in sel) for a, b in zip(i, j)])
    if mode == "involving":
        return np.array([(a in sel) or (b in sel) for a, b in zip(i, j)])
    raise ValueError(mode)

# --- usage: subset your existing beta feature matrix to set A ---
idx_A   = select_roi_indices(roi_names, DK_REGION_SETS["A"])   # -> [i_precentral-lh, i_precentral-rh]
mask_A  = edge_mask(roi_names, idx_A, mode="within")
X_A     = Xb["beta"][:, mask_A]
print("Set A ROIs:", [roi_names[i] for i in idx_A], "| edges:", X_A.shape[1])

NameError: name 'roi_names' is not defined

<br><br>